In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display
load_dotenv()

In [ ]:
google_api_key = os.getenv("GOOGLE_API_KEY")

In [ ]:
ollama_url = "http://localhost:11434/v1"

ollama_client = OpenAI(base_url=ollama_url, api_key="ollama")

gpt_model = "gpt-oss:20b"
qwen_model = "qwen2.5-coder:7b"
mistral_model = "mistra-7b"

In [ ]:
class SystemPrompts:
    alex = "You are a chatbot who is very argumentative; \
 you disagree with anything in the conversation and you challenge everything, in a snarky way. You are in conversation with a very agreeable chatbot and a very neutral chatbot."
    blake = "You are a chatbot who is very agreeable; \
 you agree with anything in the conversation and you support everything, in a very nice way. You are in conversation with a very argumentative chatbot and a very neutral chatbot."
    charlie = "You are a chatbot who is very neutral; \
 you neither agree nor disagree with anything in the conversation and you try to be as unbiased as possible. You are in conversation with a very argumentative chatbot and a very agreeable chatbot."

class Conversation:
    alex_user_prompts = """
        You are Alex, in conversation with Blake and Charlie.
        The conversation so far is as follows:
        {conversation}
        Now with this, respond with what you would like to say next, as Alex.
    """

    blake_user_prompts = """
        You are Blake, in conversation with Alex and Charlie.
        The conversation so far is as follows:
        {conversation}
        Now with this, respond with what you would like to say next, as Blake.
    """

    charlie_user_prompts = """
        You are Charlie, in conversation with Alex and Blake.
        The conversation so far is as follows:
        {conversation}
        Now with this, respond with what you would like to say next, as Charlie.
    """

In [ ]:
def build_conversation(alex_messages, blake_messages, charlie_messages):
    conversation = ""
    max_len = max(len(alex_messages), len(blake_messages), len(charlie_messages))
    for i in range(max_len):
        if i < len(alex_messages):
            conversation += f"Alex: {alex_messages[i]}\n"
        if i < len(blake_messages):
            conversation += f"Blake: {blake_messages[i]}\n"
        if i < len(charlie_messages):
            conversation += f"Charlie: {charlie_messages[i]}\n"
    return conversation

def call_alex(alex_messages, blake_messages, charlie_messages):
    conversation = build_conversation(alex_messages, blake_messages, charlie_messages)
    messages = [
        {"role": "system", "content": SystemPrompts.alex},
        {"role": "user", "content": Conversation.alex_user_prompts.format(conversation=conversation)}
    ]
    print("Messages from Alex:", messages)
    response = ollama_client.chat.completions.create(model=gpt_model, messages=messages)
    return response.choices[0].message.content

def call_blake(alex_messages, blake_messages, charlie_messages):
    conversation = build_conversation(alex_messages, blake_messages, charlie_messages)
    messages = [
        {"role": "system", "content": SystemPrompts.blake},
        {"role": "user", "content": Conversation.blake_user_prompts.format(conversation=conversation)}
    ]
    print("Messages from Blake:", messages)
    response = ollama_client.chat.completions.create(model=gpt_model, messages=messages)
    return response.choices[0].message.content

def call_charlie(alex_messages, blake_messages, charlie_messages):
    conversation = build_conversation(alex_messages, blake_messages, charlie_messages)
    messages = [
        {"role": "system", "content": SystemPrompts.charlie},
        {"role": "user", "content": Conversation.charlie_user_prompts.format(conversation=conversation)}
    ]
    print("Messages from Charlie:", messages)
    response = ollama_client.chat.completions.create(model=gpt_model, messages=messages)
    return response.choices[0].message.content

alex_messages = ["Hi"]
blake_messages = ["Hello there!"] 
charlie_messages = ["Hi there!"]

In [ ]:
display(Markdown(f"### Alex:\n{alex_messages[0]}\n"))
display(Markdown(f"### Blake:\n{blake_messages[0]}\n"))
display(Markdown(f"### Charlie:\n{charlie_messages[0]}\n"))

for i in range(5):
    alex_next = call_alex(alex_messages, blake_messages, charlie_messages)   
    display(Markdown(f"### Alex:\n{alex_next}\n"))
    alex_messages.append(alex_next)
    
    blake_next = call_blake(alex_messages, blake_messages, charlie_messages)
    display(Markdown(f"### Blake:\n{blake_next}\n"))
    blake_messages.append(blake_next)

    charlie_next = call_charlie(alex_messages, blake_messages, charlie_messages)
    display(Markdown(f"### Charlie:\n{charlie_next}\n"))
    charlie_messages.append(charlie_next)